# PnP Problem

The Perspective-n-Point or "PnP" problem refers to the common computer-vision task of estimating the 6-DOF pose of a camera given a set of 3D-2D point correspondences.

Inputs:
 - 3D locations of known points within the scene (world-space)
 - 2D locations of the scene points as observed in a camera capture (image-space)
 - camera intrinsics model

Outputs
 - position and orientation of the camera that best explains the given observations

## Geometric Intuition

A minimum of n=4 points are required to uniquely determine a solution. However, with only n=3 points, the solution set is comprised of a small number of discrete poses, from which the correct solution can usually be chosen using some heuristic based on external knowledge. For this reason P3P is often treated as the minimal form (e.g. for use in RANSAC model-fitting).

The following table describes the general solution set for n = 1 through n = 4:

| Number of points ($n \in \mathbb{N}$) | PnP solution set ($\text{camera pose} \in SE(3)$)                                                                                                                                                                                                                              |
|---------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| 1                                     | **Position**: completely unconstrained<br>**Orientation**: constrained to a 1D subspace of $SO(3)$ for a given position (rotation about the bearing ray)                                                                                                                        |
| 2                                     | **Position**: constrained to a finite 2D surface created by rotating a limaçon-type curve about the axis connecting the 2 points<br>**Orientation**: uniquely determined for a given position                                                                                          |
| 3                                     | **Position**: one of 2 discrete points resulting from the intersection of the three pairwise P2P solution sets (both on the same side of the triangle's plane, but not trivially similar to one another)<br>**Orientation**: uniquely determined for a given position |
| 4                                     | **Position**: uniquely determined<br>**Orientation**: uniquely determined                                                                                                                                                                                                       |

Note: The above table assumes that the camera chirality constraint (i.e. the requirement that the observed points appear *in front* of the imager) is enforced. If this is relaxed, the number of possible solutions is generally doubled, which is why much of the literature states that P3P actually has up to *4* possible solutions (the 2 described above, plus their reflections about the optical center).

## Algebraic Solution (P3P)

The algebraic solution to the P3P problem is a classical derivation, typically attributed to Grunert (1841). We present the full derivation here, with the help of the `sympy` computer algebra system for most of the heavy lifting.

### Notation

 - Let $X_1, X_2, X_3 \in \mathbb{R}^3$ be the world-space points
 - Let $x_1, x_2, x_3 \in \mathbb{R}^2$ be the image-space observations
 - Let $\pi: \mathbb{R}^3 \to \mathbb{R}^2$ be the camera projection function
 - Let $T = [R \mid t] \in SE(3)$ be the 6-DOF pose of the camera in world-space

The P3P problem is to find $T = [R \mid t]$ such that:

$$\pi(T^{-1} X_i) = x_i$$

for $i = 1, 2, 3$.

### 1) Tetrahedral Construction

Consider the tetrahedon formed by the camera center (which we denote $P$) and the 3 world-space points.

Denote the lengths of the tetrahedron's base:
 - $s_{12} = \lVert X_1 - X_2 \rVert$
 - $s_{13} = \lVert X_1 - X_3 \rVert$
 - $s_{23} = \lVert X_2 - X_3 \rVert$

Denote the lengths of the tetrahedron's legs (the distances from camera center to world point):
 - $d_1 = \lVert P - X_1 \rVert$
 - $d_2 = \lVert P - X_2 \rVert$
 - $d_3 = \lVert P - X_3 \rVert$

Denote the angles subtended by the tetrahedron's legs:
 - $\theta_{12} = \angle X_1PX_2$
 - $\theta_{13} = \angle X_1PX_3$
 - $\theta_{23} = \angle X_2PX_3$

The $s_{ij}$ can be computed directly from the givens. The $\theta_{ij}$ can also be derived from the givens by taking the bearing-vectors $\pi^{-1}(x_i)$ and applying the usual angle-between-vectors formula:

$$\cos \theta_{ij} = \pi^{-1}(x_i) \cdot \pi^{-1}(x_j)$$

This leaves only the $d_i$ as the 3 unknowns. Once these are obtained, the full geometry is known and the actual camera pose can be directly computed via rigid-body transform.

### 2) Law-of-Cosines Constraints

Consider the three triangular faces involving $P$. From the law-of-cosines, we have the following constraints on the $d_i$:

 - $s_{12} = d_1^2 + d_2^2 - 2d_1d_2\cos \theta_{12}$
 - $s_{13} = d_1^2 + d_3^2 - 2d_1d_3\cos \theta_{13}$
 - $s_{23} = d_2^2 + d_3^2 - 2d_2d_3\cos \theta_{23}$

This is a system of thee polynomial equations in three unknowns. 

### 3) Change of Variables

At this point, the literature typically makes a change of  

TODO: $u = d_2/d_1$, $v = d_3/d_1$

### 4) Reduction to a Quartic in a Single Variable

Construct the tetrahedron
Law-of-cosines
System of quadratics in d1, d2, d3
Reduction to quartic one variable
Numerical solution

In [1]:
import sympy

ModuleNotFoundError: No module named 'sympy'

In [ ]:
a, b, c = sympy.symbols("a b c")
r1, r2, r3 = sympy.symbols("r_1 r_2 r_3")
cos_alpha, cos_beta, cos_gamma = sympy.symbols("cos_alpha cos_beta cos_gamma")

In [ ]:
def system_to_polynomial(
    system: list[sympy.Expr],
    vars: list[sympy.Symbol],
    keep: list[sympy.Symbol] | None = None,
):
    """
    Convert a polynomial system into a single polynomial via elimination.

    Parameters
    ----------
    system : list of sympy Expr
        Polynomial equations assumed equal to zero.
    vars : list of sympy Symbol
        All variables in the system, ordered for elimination.
    keep : list of sympy Symbol, optional
        Variables to keep (default: keep the first variable only).

    Returns
    -------
    sympy Expr
        Single polynomial in the kept variables.
    """
    vars = list(vars)

    if keep is None:
        keep = [vars[-1]]
    else:
        keep = list(keep)

    eliminate = [v for v in vars if v not in keep]

    # Groebner basis
    G = sympy.groebner(system, *vars, order="lex")

    print(G)

    # Elimination theorem: select polynomials free of eliminated vars
    polys = [
        sympy.factor(g)
        for g in G
        if g.free_symbols.isdisjoint(eliminate)
    ]

    if not polys:
        raise ValueError("Elimination produced no polynomial")

    # Canonical choice: lowest total degree
    polys.sort(key=lambda p: sympy.Poly(p, *keep).total_degree())

    return polys[0]

In [ ]:
x, y, z = sympy.symbols('x y z')

system = [
    x + y + z - 1,
    x**2 + y - z,
    y**2 - x,
]

poly_x = system_to_polynomial(system, [x, y, z])
print(poly_x)

GroebnerBasis([2*x - z**3 + 5*z**2 - 12*z + 4, 2*y + z**3 - 5*z**2 + 14*z - 6, z**4 - 6*z**3 + 18*z**2 - 16*z + 4], x, y, z, domain='ZZ', order='lex')
z**4 - 6*z**3 + 18*z**2 - 16*z + 4


In [ ]:
import sympy as sp

# Unknowns
u, v, d1 = sp.symbols("u v d1", real=True)

# Known geometry
a, b, c = sp.symbols("a b c", positive=True)
ca, cb, cg = sp.symbols("cosa cosb cosg", real=True)

# Law-of-cosines
eq1 = (u*d1)**2 + (v*d1)**2 - 2*u*v*d1**2*ca - a**2
eq2 = d1**2 + (v*d1)**2 - 2*v*d1**2*cb - b**2
eq3 = d1**2 + (u*d1)**2 - 2*u*d1**2*cg - c**2

# Remove scale
eq1 /= d1**2
eq2 /= d1**2
eq3 /= d1**2

eq1 = sympy.simplify(eq1)
eq2 = sympy.simplify(eq2)
eq3 = sympy.simplify(eq3)

print(eq1)
print(eq2)
print(eq3)

vars = [v, u]

# Eliminate v → Grunert quartic in u
quartics = system_to_polynomial(
    [eq1, eq2, eq3],
    vars=[v, u],
    keep=[u]
)

assert len(quartics) > 0
quartic = quartics[0]

quartic

-a**2/d1**2 - 2*cosa*u*v + u**2 + v**2
-b**2/d1**2 - 2*cosb*v + v**2 + 1
-c**2/d1**2 - 2*cosg*u + u**2 + 1


In [ ]:
import sympy as sp

# Unknowns
u, v = sp.symbols("u v")

# Parameters
a, b, c = sp.symbols("a b c", positive=True)
ca, cb, cg = sp.symbols("ca cb cg", real=True)

# Normalized equations (already divided by d1^2)
eq1 = u**2 + v**2 - 2*u*v*ca - a**2
eq2 = 1 + v**2 - 2*v*cb - b**2
eq3 = 1 + u**2 - 2*u*cg - c**2

# Step 1: eliminate v using eq1 & eq2
res_v = sp.resultant(eq1, eq2, v)

# Step 2: eliminate scale using eq3
quartic = sp.resultant(res_v, eq3, u)

quartic = sp.factor(quartic)
display(quartic)

sp.Poly(quartic, u)

a**8 - 4*a**6*b**2 - 4*a**6*c**2 + 8*a**6*ca*cb*cg - 8*a**6*cb**2 - 8*a**6*cg**2 + 8*a**6 + 6*a**4*b**4 - 8*a**4*b**2*c**2*ca**2 + 12*a**4*b**2*c**2 - 16*a**4*b**2*ca**2*cg**2 + 8*a**4*b**2*ca**2 - 8*a**4*b**2*ca*cb*cg + 16*a**4*b**2*cb**2 + 24*a**4*b**2*cg**2 - 24*a**4*b**2 + 6*a**4*c**4 - 16*a**4*c**2*ca**2*cb**2 + 8*a**4*c**2*ca**2 - 8*a**4*c**2*ca*cb*cg + 24*a**4*c**2*cb**2 + 16*a**4*c**2*cg**2 - 24*a**4*c**2 + 16*a**4*ca**2*cb**2 + 16*a**4*ca**2*cg**2 - 8*a**4*ca**2 - 32*a**4*ca*cb**3*cg - 32*a**4*ca*cb*cg**3 + 16*a**4*ca*cb*cg + 16*a**4*cb**4 + 48*a**4*cb**2*cg**2 - 40*a**4*cb**2 + 16*a**4*cg**4 - 40*a**4*cg**2 + 24*a**4 - 4*a**2*b**6 + 16*a**2*b**4*c**2*ca**2 - 12*a**2*b**4*c**2 + 32*a**2*b**4*ca**2*cg**2 - 16*a**2*b**4*ca**2 - 8*a**2*b**4*ca*cb*cg - 8*a**2*b**4*cb**2 - 24*a**2*b**4*cg**2 + 24*a**2*b**4 + 16*a**2*b**2*c**4*ca**2 - 12*a**2*b**2*c**4 + 32*a**2*b**2*c**2*ca**3*cb*cg - 64*a**2*b**2*c**2*ca**2 + 48*a**2*b**2*c**2*ca*cb*cg - 32*a**2*b**2*c**2*cb**2 - 32*a**2*b**2*c**2

Poly(a**8 - 4*a**6*b**2 - 4*a**6*c**2 + 8*a**6*ca*cb*cg - 8*a**6*cb**2 - 8*a**6*cg**2 + 8*a**6 + 6*a**4*b**4 - 8*a**4*b**2*c**2*ca**2 + 12*a**4*b**2*c**2 - 16*a**4*b**2*ca**2*cg**2 + 8*a**4*b**2*ca**2 - 8*a**4*b**2*ca*cb*cg + 16*a**4*b**2*cb**2 + 24*a**4*b**2*cg**2 - 24*a**4*b**2 + 6*a**4*c**4 - 16*a**4*c**2*ca**2*cb**2 + 8*a**4*c**2*ca**2 - 8*a**4*c**2*ca*cb*cg + 24*a**4*c**2*cb**2 + 16*a**4*c**2*cg**2 - 24*a**4*c**2 + 16*a**4*ca**2*cb**2 + 16*a**4*ca**2*cg**2 - 8*a**4*ca**2 - 32*a**4*ca*cb**3*cg - 32*a**4*ca*cb*cg**3 + 16*a**4*ca*cb*cg + 16*a**4*cb**4 + 48*a**4*cb**2*cg**2 - 40*a**4*cb**2 + 16*a**4*cg**4 - 40*a**4*cg**2 + 24*a**4 - 4*a**2*b**6 + 16*a**2*b**4*c**2*ca**2 - 12*a**2*b**4*c**2 + 32*a**2*b**4*ca**2*cg**2 - 16*a**2*b**4*ca**2 - 8*a**2*b**4*ca*cb*cg - 8*a**2*b**4*cb**2 - 24*a**2*b**4*cg**2 + 24*a**2*b**4 + 16*a**2*b**2*c**4*ca**2 - 12*a**2*b**2*c**4 + 32*a**2*b**2*c**2*ca**3*cb*cg - 64*a**2*b**2*c**2*ca**2 + 48*a**2*b**2*c**2*ca*cb*cg - 32*a**2*b**2*c**2*cb**2 - 32*a**2*b**2

In [ ]:
import sympy as sp

# Unknowns (distances to the 3 points)
d1, d2, d3 = sp.symbols("d1 d2 d3", real=True, positive=True)

# Known world triangle sides
s12, s13, s23 = sp.symbols("s12 s13 s23", positive=True)

# Known angles between bearing vectors
m12, m13, m23 = sp.symbols("m12 m13 m23", real=True)  # cos(alpha), cos(beta), cos(gamma)

# -------------------------------
# 1. Law-of-cosines equations
# -------------------------------
eq1 = d1**2 + d2**2 - 2*d1*d2*m12 - s12**2
eq2 = d1**2 + d3**2 - 2*d1*d3*m13 - s13**2
eq3 = d2**2 + d3**2 - 2*d2*d3*m23 - s23**2

# -------------------------------
# 2. Normalize by d1
# -------------------------------
# Define ratios u = d2/d1, v = d3/d1
u, v = sp.symbols("u v", real=True, positive=True)
subs = {d2: u*d1, d3: v*d1}

eq1_normalized = eq1.subs(subs)
eq2_normalized = eq2.subs(subs)
eq3_normalized = eq3.subs(subs)

eq1_normalized /= d1**2
eq2_normalized /= d1**2
eq3_normalized /= d1**2

# use one of the eqs to get an expression for d1 in terms of u and v
d1sq_expr = sp.solve(eq2_normalized, d1**2)[0]
print(d1sq_expr)

eq1_normalized = eq1_normalized.subs({d1**2: d1sq_expr})
eq3_normalized = eq3_normalized.subs({d1**2: d1sq_expr})

eq1_normalized = sp.simplify(eq1_normalized)
eq3_normalized = sp.simplify(eq3_normalized)

print(eq1_normalized)
print(eq3_normalized)

# -------------------------------
# 3. Eliminate v using resultant to get quartic in u
# -------------------------------
quartic_u = sp.resultant(eq1_normalized, eq3_normalized, v)
quartic_u *= s13**4     # simplify

quartic_u = sp.factor(quartic_u)

print("Grunert quartic in u (symbolic):")
display(quartic_u)
sp.Poly(quartic_u, u)

s13**2/(-2*m13*v + v**2 + 1)
(-2*m12*s13**2*u + s12**2*(2*m13*v - v**2 - 1) + s13**2*u**2 + s13**2)/s13**2
(-2*m23*s13**2*u*v + s13**2*u**2 + s13**2*v**2 + s23**2*(2*m13*v - v**2 - 1))/s13**2
Grunert quartic in u (symbolic):


4*m12**2*s13**4*u**2 - 8*m12**2*s13**2*s23**2*u**2 + 4*m12**2*s23**4*u**2 + 8*m12*m13**2*s12**2*s23**2*u - 8*m12*m13*m23*s12**2*s13**2*u**2 - 8*m12*m13*m23*s12**2*s23**2*u**2 + 8*m12*m23**2*s12**2*s13**2*u**3 - 4*m12*s12**2*s13**2*u**3 + 4*m12*s12**2*s13**2*u + 4*m12*s12**2*s23**2*u**3 - 4*m12*s12**2*s23**2*u - 4*m12*s13**4*u**3 - 4*m12*s13**4*u + 8*m12*s13**2*s23**2*u**3 + 8*m12*s13**2*s23**2*u - 4*m12*s23**4*u**3 - 4*m12*s23**4*u + 4*m13**2*s12**4*u**2 - 4*m13**2*s12**2*s23**2*u**2 - 4*m13**2*s12**2*s23**2 - 4*m13*m23*s12**4*u**3 - 4*m13*m23*s12**4*u + 4*m13*m23*s12**2*s13**2*u**3 + 4*m13*m23*s12**2*s13**2*u + 4*m13*m23*s12**2*s23**2*u**3 + 4*m13*m23*s12**2*s23**2*u + 4*m23**2*s12**4*u**2 - 4*m23**2*s12**2*s13**2*u**4 - 4*m23**2*s12**2*s13**2*u**2 + s12**4*u**4 - 2*s12**4*u**2 + s12**4 + 2*s12**2*s13**2*u**4 - 2*s12**2*s13**2 - 2*s12**2*s23**2*u**4 + 2*s12**2*s23**2 + s13**4*u**4 + 2*s13**4*u**2 + s13**4 - 2*s13**2*s23**2*u**4 - 4*s13**2*s23**2*u**2 - 2*s13**2*s23**2 + s23**4*u**4 + 

Poly((-4*m23**2*s12**2*s13**2 + s12**4 + 2*s12**2*s13**2 - 2*s12**2*s23**2 + s13**4 - 2*s13**2*s23**2 + s23**4)*u**4 + (8*m12*m23**2*s12**2*s13**2 - 4*m12*s12**2*s13**2 + 4*m12*s12**2*s23**2 - 4*m12*s13**4 + 8*m12*s13**2*s23**2 - 4*m12*s23**4 - 4*m13*m23*s12**4 + 4*m13*m23*s12**2*s13**2 + 4*m13*m23*s12**2*s23**2)*u**3 + (4*m12**2*s13**4 - 8*m12**2*s13**2*s23**2 + 4*m12**2*s23**4 - 8*m12*m13*m23*s12**2*s13**2 - 8*m12*m13*m23*s12**2*s23**2 + 4*m13**2*s12**4 - 4*m13**2*s12**2*s23**2 + 4*m23**2*s12**4 - 4*m23**2*s12**2*s13**2 - 2*s12**4 + 2*s13**4 - 4*s13**2*s23**2 + 2*s23**4)*u**2 + (8*m12*m13**2*s12**2*s23**2 + 4*m12*s12**2*s13**2 - 4*m12*s12**2*s23**2 - 4*m12*s13**4 + 8*m12*s13**2*s23**2 - 4*m12*s23**4 - 4*m13*m23*s12**4 + 4*m13*m23*s12**2*s13**2 + 4*m13*m23*s12**2*s23**2)*u - 4*m13**2*s12**2*s23**2 + s12**4 - 2*s12**2*s13**2 + 2*s12**2*s23**2 + s13**4 - 2*s13**2*s23**2 + s23**4, u, domain='ZZ[s12,s13,s23,m12,m13,m23]')

## General Solution

RANSAC

Non-linear least squares manifold optimization
